In [1]:

from urllib.parse import urljoin
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
import os
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
from langchain_core.messages import SystemMessage, HumanMessage
import sqlite3
import pandas as pd
import json

from typing import TypedDict
from pydantic import BaseModel
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END



/Users/caineosborne/Projects2026/demographics-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
memory = MemorySaver()

load_dotenv(override=True)
api_key=os.getenv("OPENROUTER_API_KEY")

In [3]:
from tavily import TavilyClient

client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])



In [4]:
news = client.search(
    query=(
        '"population" OR "births" OR "fertility" OR '
        '"deaths" OR "net migration" statistics released'
    ),
    topic="news",
    time_range="day",
    search_depth="basic",
    max_results=10,
)

official = client.search(
    query=(
        '"population" OR "births" OR "fertility" OR '
        '"deaths" OR "migration" "statistical release"'
    ),
    topic="general",
    time_range="week",
    search_depth="basic",
    max_results=10,
)

government = client.search(
    query=(
        '"population" OR "births" OR "fertility" OR '
        '"deaths" OR "migration" '
        '"national statistics" "2026"'
    ),
    topic="general",
    time_range="week",
    search_depth="basic",
    max_results=10,
    include_domains=[
        "gov.au",
        "gov.uk",
        "govt.nz",
        "gov",
        "gc.ca",
        "go.jp",
        "gov.cn",
    ],
)

In [5]:
results = (
    news["results"]
    + official["results"]
    + government["results"]
)

In [6]:
SYSTEM_PROMPT = """
You are a demographic release reviewer.

Your task is to review search results and identify newly released
national demographic statistics.

Relevant topics:
- Population
- Births
- Deaths
- Fertility
- Migration

Include:
- Official statistical releases
- News articles reporting new national demographic statistics

Exclude:
- Commentary and opinion
- Generic information pages
- Historical archives
- Regional-only statistics
- Unrelated uses of words such as "migration" or "deaths"

For each relevant result, return:
- title
- url
- reason
- needs_full_page

Set needs_full_page to true when the snippet does not contain
enough information to extract the key statistics.

Return only JSON. Do not summarise the dataset, offer next steps,
or ask the user questions.

A result is relevant only if it reports a NEWLY RELEASED or newly updated
national demographic statistic.

The result should contain new factual information worth reporting, such as:
- a new population estimate
- newly released birth or death totals
- a newly released fertility rate
- new migration figures
- a meaningful revision to previous demographic statistics

Exclude:
- future scheduled releases that have not happened yet
- generic data portals or topic pages
- old releases returned by search
- population forecast/reference websites
- articles that merely discuss demographic trends
- regional statistics unless they represent the country's official national release
- routine weekly mortality surveillance unless it represents a meaningful demographic development
- low-quality sites repeating statistics from another source when no new information is added

Ask: "Would this be worth mentioning in today's demographic briefing?"
If no, exclude it.


"""

In [7]:
class RelevantResult(BaseModel):
    title: str
    url: str
    reason: str
    needs_full_page: bool


class ReviewOutput(BaseModel):
    relevant_results: list[RelevantResult]

    

In [8]:
llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    model="qwen/qwen3.7-flash",
)

review_llm = llm.with_structured_output(ReviewOutput)

In [9]:
class State(TypedDict):
    search_results: list[dict]
    relevant_results: list[dict]

In [10]:
def review_results(state: State):

    response = review_llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(
            content=json.dumps(
                state["search_results"],
                ensure_ascii=False
            )
        ),
    ])

    return {
        "relevant_results": [
            result.model_dump()
            for result in response.relevant_results
        ]
    }

In [11]:
graph_builder = StateGraph(State)

graph_builder.add_node(
    "review_results",
    review_results
)

graph_builder.add_edge(
    START,
    "review_results"
)

graph_builder.add_edge(
    "review_results",
    END
)

graph = graph_builder.compile()

In [12]:
output = graph.invoke({
    "search_results": results,
    "relevant_results": [],
})

In [16]:
output['relevant_results']

[{'title': 'UK Population Shift: Deaths to Outnumber Births from 2026! (2026)',
  'url': 'https://eclipsesoccer.org/article/uk-population-shift-deaths-to-outnumber-births-from-2026',
  'reason': 'Reports newly revised UK population projections from the ONS, including downward revisions to long-term growth forecasts, shifting age structures, and the role of declining fertility and net migration.',
  'needs_full_page': False},
 {'title': 'Deaths registered weekly in England and Wales, provisional',
  'url': 'https://www.ons.gov.uk/releases/deathsregisteredweeklyinenglandandwalesprovisionalweekending25september2026',
  'reason': 'Official ONS statistical release providing provisional weekly mortality figures for England and Wales.',
  'needs_full_page': True},
 {'title': 'Reason for international migration : International students update December 2026',
  'url': 'https://www.gov.uk/government/statistics/announcements/reason-for-international-migration-international-students-update-decembe

In [17]:
output['search_results']

[{'url': 'https://www.indiasnews.net/news/279289414/eci-eliminated-15-of-india-population-ex-cec-quraishi-takes-dig-at-sir-questions-illegal-immigrant-numbers',
  'title': '"ECI eliminated 15% of India\'s population": Ex-CEC Quraishi takes dig at SIR, questions illegal immigrant numbers',
  'content': 'India\'s News.Net\n\n###### "Meeting will finalise way ahead": Odisha Dy CM KV Singh Deo outlines roadmap at BJP State Executive Meet in Cuttack\n\nIndia\'s News.Net\n\n###### Himachal Governor confers state awards on 32 teachers; hails quality, value-based education for Viksit Bharat 2047\n\nIndia\'s News.Net\n\n###### Nepal floods: Death toll reaches 1,341 as police release new figures\n\nIndia\'s News.Net [...] India\'s News.Net\n\n###### India\'s headline inflation expected to hit 4.88% in August on food, fuel pressures: Report\n\nIndia\'s News.Net\n\n###### Gujarat Police takes major decision to eradicate the drug menace;dedicated \'Anti-Narcotics Wing\' to become operational\n\nInd